## Project 2: Tidying and Transforming Wide Datasets 
## IS 362 Data Acquisition and Management

## Dataset 1: Book Sales
This dataset tracks book sales over a three month period. It is currently in a wide format because the months (January Sales, February Sales, March Sales) are listed as individual columns rather than data values under a single Month variable.

We will  melt the month columns into a single Month column, store the numeric values in a Sales_Count column, and take away the sales from the end of the months to debloat the data

In [ ]:
pip install pandas

In [4]:
import pandas as pd

In [ ]:
# getting the data
df_books = pd.read_csv('books.csv')
print("--- Original Bookstore Data ---")
display(df_books)

# tidying the dataset 
tidy_books = df_books.melt(
    id_vars=['Book Title', 'Genre'], 
    value_vars=['January Sales', 'February Sales', 'March Sales'],
    var_name='Month', 
    value_name='Sales_Count'
)

# debloating the data
tidy_books['Month'] = tidy_books['Month'].str.replace(' Sales', '')

print("\n--- Cleaned bookstore Data ---")
display(tidy_books.head())

# total sales by genre
print("\n--- Top selling books ---")
top_books = tidy_books.groupby('Book Title')['Sales_Count'].sum().reset_index()
display(top_books.sort_values('Sales_Count', ascending=False))

print("\n--- Top selling genres ---")
top_genres = tidy_books.groupby('Genre')['Sales_Count'].sum().reset_index()
display(top_genres.sort_values('Sales_Count', ascending=False))

print("\n--- Monlthy sales count ---")
# converting the months into a categorical type so it goes chronologically instead of alphabetically
monthly = tidy_books.groupby('Month')['Sales_Count'].sum().reset_index()
monthly['Month'] = pd.Categorical(monthly['Month'], categories=['January', 'February', 'March'], ordered=True)
display(monthly.sort_values('Month'))

print("\n--- Romance books comparison over time ---")
romance = tidy_books[tidy_books['Genre'] == 'Romance']
romance_pivot = romance.pivot(index='Month', columns='Book Title', values='Sales_Count')
romance_pivot.index = pd.Categorical(romance_pivot.index, categories=['January', 'February', 'March'], ordered=True)
display(romance_pivot.sort_index())

Bookstore Analysis Conclusion:
The clean data allows us to easily look through the sales figures. Overall, The Last Summer was the best selling individual book, driving Romance to be the most popular genre overall. The store saw a steady month over month increase in total sales, growing from 194 units in January to 252 units in March. When comparing the two romance novels directly, The Last Summer consistently outsold Second Chances every single month.

## Dataset 2: Brooklyn Demographics
This dataset tracks Brooklyn's population across 1920, 1970, and 2010. its a bit untidy because it mixes absolute population counts and calculated percentages horizontally across the row for every demographic group, which can be too much to look at.

We will isolate the absolute population counts and melt them into a Demographic_Group column and a Count column, dropping historical null values where certain demographics were not tracked.

In [ ]:
# loading the data
df_demo = pd.read_csv('demographics.csv')
df_demo['Borough'] = 'Brooklyn'
print("--- Original Demographic Data ---")
display(df_demo)

# tidying the data: selecting only the count columns to melt
count_columns = ['White', 'Non-Hispanic Whites', 'Black', 'Asian', 'Other or Mixed', 'Hispanic/Latino']

tidy_demo = df_demo.melt(
    id_vars=['Borough', 'Year', 'Population'], 
    value_vars=count_columns,
    var_name='Demographic_Group', 
    value_name='Count'
)

# dropping the NaN values to convert to integer values
tidy_demo = tidy_demo.dropna(subset=['Count'])
tidy_demo['Count'] = tidy_demo['Count'].astype(int)

print("\n--- Tidied Demographic Data ---")
display(tidy_demo.sort_values(by=['Year', 'Demographic_Group']).head())

# Analysis: total shift in borough demogrpahic
demo_pivot = tidy_demo.pivot(index='Year', columns='Demographic_Group', values='Count').fillna(0)
display(demo_pivot)

print("\n--- Total shift by demographic group ---")
# calculates the absolute shift (max count minus min count) to see which demographic changed the most
shift = demo_pivot.max() - demo_pivot.min()
shift_df = shift.sort_values(ascending=False).reset_index(name='Total_Shift_Count')
display(shift_df)

Demographic shift  analysis conclusion:
By melting the dataset and removing the initial N/A values, it isolates the pure population counts for each demographic group. Because categories like Hispanic/Latino and Non-Hispanic Whites were not tracked in the 1920 and 1970 census years, filled those missing historical data points with zeros in the table to accurately measure their emergence. By calculating the absolute difference between the maximum and minimum population counts across the century,it reveals that the White demographic experienced the largest overall change shifting by over 912,000 residents, closely followed by the Black demographic, which saw the largest pure growth, increasing by over 828,000 residents.

## Dataset 3: GPU Benchmarks
This hardware dataset compares three graphics cards. its untidy because the column headers (1080p Medium (FPS), 1440p Ultra (FPS),) are specific graphical settings containing performance metrics, rather than distinct variables.

We will melt the resolution settings into a Resolution_Setting column and map the intersecting numbers into a designated Average_FPS column.

In [ ]:
# loading the data
df_gpu = pd.read_csv('gpus.csv')
print("--- Original GPU Data ---")
display(df_gpu)

# tidying the data
tidy_gpu = df_gpu.melt(
    id_vars=['GPU Model'], 
    var_name='Resolution_Setting', 
    value_name='Average_FPS'
)

print("\n--- Cleaned GPU data ---")
display(tidy_gpu.head())

# finding the performance drop off for each GPU
gpu_performance = tidy_gpu.groupby('GPU Model')['Average_FPS'].agg(
    Max_FPS='max', 
    Min_FPS='min'
).reset_index()

# calculateing the drop from the easiest to hardest setting
gpu_performance['FPS_Drop'] = gpu_performance['Max_FPS'] - gpu_performance['Min_FPS']

print("\n--- Performance Drop off (1080p Medium to 4K Ultra) ---")
display(gpu_performance.sort_values(by='FPS_Drop', ascending=False))

GPU benchmarks result:
melting the resolutions into a single column, each row now represents one distinct pairing (one GPU at one specific resolution). Grouping by the GPU Model allows us to calculate the absolute FPS drop off between the least demanding setting and the most demanding setting. The final result reveals that while the RTX 4090 outputs the highest overall frames, it also experiences the largest absolute drop off (122.7 FPS) when resolution is altered or maxamized.